# ROP分類: Train-Val-Test 5-Fold CV (clinical_v3_tvt)

## 概要

clinical_v3モデル（EfficientNet-B0 + 臨床データ融合、5タスク同時学習）のCVをtrain-val-test分割に変更する。
元のclinical_v3ではval setをearly stopping（モデル選択）と最終評価の両方に使用しており、楽観的バイアスのリスクがある。
本Notebookではval setをearly stopping専用、test setを最終評価専用に分離する。

---

## clinical_v3 からの変更点

| 項目 | clinical_v3 | clinical_v3_tvt (本Notebook) |
|------|------------|------------------------------|
| Train / Val / Test | 80% / 20% / (なし) | **60% / 20% / 20%** |
| Early stopping | Val loss | Val loss (同一) |
| 最終評価 | Val fold の予測 | **Test fold の予測** |
| モデル・Loss・Augmentation | - | **変更なし** |
| ハイパーパラメータ | - | **変更なし** |

---

## モデルアーキテクチャ

```
Input Image (512x512)           Clinical Features (4-dim)
       |                               |
 EfficientNet-B0                 ClinicalEncoder MLP
 (ImageNet pretrained)           4 → 64 → ReLU → BN → 32
       |                                |
 Image Features (1280)                  |
       |                                |
       +---------- Concatenate ---------+
                      |
               Fused (1312-dim)
                      |
          +----+----+----+----+----+
        Zone Stage Plus  AROP Treatment
        (3)  (4)   (3)  (2)    (2)
```

### 臨床特徴量
| Feature | Type | 前処理 |
|---------|------|--------|
| Sex | Binary (0/1) | 正規化なし |
| GA (weeks) | Continuous | Z-score (Train foldのmean/std) |
| BW (g) | Continuous | Z-score (Train foldのmean/std) |
| PMA (weeks) | Continuous | Z-score (Train foldのmean/std) |

### 学習設定
- **Optimizer**: AdamW (lr=1e-4, weight_decay=1e-3)
- **Scheduler**: OneCycleLR
- **Loss**: Weighted Focal Loss + Cross Entropy (平均)
- **Label smoothing**: 0.1
- **MixUp**: alpha=0.2, p=0.5
- **Early stopping**: patience=15 (Val loss)
- **Max epochs**: 200

---

## CV Split Design

`StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` で5 foldを事前生成し、
テストfoldをローテーションする。同一video_idが異なるsplitに分かれないことを保証。

| Iteration | Train (60%) | Val (20%) | Test (20%) |
|-----------|------------|-----------|------------|
| 1 | Fold 3,4,5 | Fold 2 | **Fold 1** |
| 2 | Fold 4,5,1 | Fold 3 | **Fold 2** |
| 3 | Fold 5,1,2 | Fold 4 | **Fold 3** |
| 4 | Fold 1,2,3 | Fold 5 | **Fold 4** |
| 5 | Fold 2,3,4 | Fold 1 | **Fold 5** |

- **Clinical stats (mean/std/median)**: Train 3 folds のみから計算
- **Class weights**: Train 3 folds のみから計算
- **Early stopping**: Val fold の loss で判定
- **最終予測**: Test fold のみ → predictions.csv（各画像が1回だけ評価される）

---

## 評価内容

### 1. Per-Image メトリクス (Cell 6)
Test fold予測（5 iteration分を結合）から、fold間の平均±SDを算出。
- **Multiclass** (Zone/Stage/Plus): Accuracy, Quadratic Kappa, F1-macro
- **Binary** (AROP/Treatment): Accuracy, Kappa, F1-macro, Sensitivity, AUC

### 2. Top-10/Top-5 画像選出 (Cell 7)
品質特徴量に基づきvideo_idごとに上位画像を選出（C-rule3_thr0.80）。
- Stage 1: `disc_edge_coverage_ratio >= 0.80` → `0.4*retina_norm + 0.4*grad_norm + 0.2*mbss_norm` 降順
- Stage 2 (Fallback): Stage1候補 < K枚の場合、`retina_ratio` 降順で補完

### 3. Video-Level Majority Vote (Cell 8)
Top-10/Top-5 画像の予測をvideo_id単位で集約し、患者単位の分類性能を評価。
- **Hard vote**: 予測クラスの最頻値（mode）
- **Soft vote**: 確率の平均 → argmax / threshold
- **RW-ROP** (Referral-Warranted ROP): `P(RW-ROP) = 1 - (1-P(Plus)) * (1-P(Stage3)) * (1-P(ZoneI))`
- 6条件を比較: All/Top-10/Top-5 × Hard/Soft

### 4. Threshold Optimization (Cell 9)
Treatment / RW-ROP について3つの動作点を報告。
- Default (0.5)
- Sensitivity >= 95%（スクリーニング用途）
- Youden's J 最大点

### 5. Vote Concordance Analysis (Cell 10)
Top-10/Top-5の各動画について、投票一致率（majority classの得票比率）を計算。
一致率の区間別（50-60%, 60-70%, ..., 90-100%）に分類正解率を集計し、
モデル予測の信頼度指標としての有用性を検証。

### 6. Results Summary (Cell 11)
- 全メトリクスの一覧表示
- clinical_v3 (val=test) との比較表

---

## 出力ファイル

```
Article/outputs_clinical_v3_tvt/
├── fold_1/ ~ fold_5/
│   └── best_model.pt          # モデル重み + 臨床統計量
├── predictions.csv            # 全テスト予測（~6,448行、各画像1回）
├── config.json                # 設定 + per-image metrics
├── top10_selected_images.csv  # Top-10選出画像リスト
├── majority_vote_results.json # Video-level集約結果
└── top10_evaluation_results.json # 閾値最適化結果
```

In [1]:
# ==================== Cell 1: Imports & Config ====================
import os
import sys
import json as json_lib
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

import timm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, classification_report,
    cohen_kappa_score, accuracy_score, f1_score, roc_curve
)

import albumentations as A
from albumentations.pytorch import ToTensorV2

from datetime import datetime
from scipy import stats

def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


class Config:
    """Training configuration — identical to clinical_v3 except OUTPUT_DIR."""

    DATA_ROOT = Path(r'E:\Multicenter_ROP_study')
    KUBOTA_DIR = DATA_ROOT / 'Multicenter_images' / 'Kubota_selection'
    EXCEL_PATH = DATA_ROOT / 'multicenter_patient_data_20260127.xlsx'
    OUTPUT_DIR = Path(r'C:\Users\ykita\ROP_AI_project\Article\outputs_clinical_v3_tvt')

    MODEL_NAME = 'efficientnet_b0'
    PRETRAINED = True
    DROPOUT = 0.5

    N_CLINICAL_FEATURES = 4
    CLINICAL_EMBED_DIM = 32

    IMG_SIZE = 512

    BATCH_SIZE = 16
    NUM_WORKERS = 0
    EPOCHS = 200
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-3
    PATIENCE = 15

    N_FOLDS = 5

    TASK_WEIGHTS = {
        'zone': 1.0,
        'stage': 1.0,
        'plus': 1.0,
        'aggressive_rop': 1.5,
        'treatment': 1.5
    }

    LABEL_SMOOTHING = 0.1

    MIXUP_ALPHA = 0.2
    MIXUP_P = 0.5

    USE_CLASS_WEIGHTS = True
    USE_FOCAL_LOSS = True
    FOCAL_GAMMA = 2.0
    USE_WEIGHTED_SAMPLER = False

    QUALITY_FILTER = ['Good', 'Fair']

    VERSION = 'clinical_v3_tvt'

config = Config()
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {config.OUTPUT_DIR}')

Using device: cuda
GPU: Quadro RTX 5000
Output directory: C:\Users\ykita\ROP_AI_project\Article\outputs_clinical_v3_tvt


In [2]:
# ==================== Cell 2: Data Loading ====================

def load_quality_labeled_images(kubota_dir: Path) -> pd.DataFrame:
    """Load images from Kubota_selection folder."""
    data = []
    for quality in ['Good', 'Fair', 'Bad', 'Worst']:
        folder = kubota_dir / quality
        if not folder.exists():
            continue
        for img_path in folder.glob('*.png'):
            if img_path.stat().st_size == 0:
                continue
            filename = img_path.stem
            parts = filename.rsplit('_', 1)
            video_id = parts[0] if len(parts) >= 2 else filename
            data.append({
                'image_path': str(img_path),
                'video_id': video_id,
                'quality': quality
            })
    return pd.DataFrame(data)


def load_patient_data(excel_path: Path) -> pd.DataFrame:
    """Load and preprocess patient data including clinical features."""
    df = pd.read_excel(excel_path)

    column_mapping = {
        'video_id': df.columns[0],
        'sex': df.columns[2],
        'ga': df.columns[3],
        'bw': df.columns[4],
        'pma': df.columns[5],
        'zone': df.columns[6],
        'stage': df.columns[7],
        'plus': df.columns[8],
        'aggressive_rop': df.columns[9],
        'treatment': df.columns[12],
    }
    rename_dict = {v: k for k, v in column_mapping.items()}
    df = df.rename(columns=rename_dict)

    df['zone_label'] = df['zone'].apply(lambda x: int(x) - 1 if pd.notna(x) and str(x).isdigit() else -1)
    df['stage_label'] = df['stage'].apply(lambda x: int(x) if pd.notna(x) and str(x).isdigit() else -1)

    def map_plus_label(x):
        if pd.isna(x):
            return -1
        if isinstance(x, (int, float)) and not np.isnan(x):
            val = int(x)
            if val in [0, 1, 2]:
                return val
        text_map = {'normal': 0, 'preplus': 1, 'plus': 2, '0': 0, '1': 1, '2': 2}
        return text_map.get(str(x).lower().strip(), -1)

    df['plus_label'] = df['plus'].apply(map_plus_label)
    df['aggressive_rop_label'] = df['aggressive_rop'].apply(
        lambda x: 1 if str(x).lower().strip() in ['yes', 'y', '1', 'true'] else 0)
    df['treatment_label'] = df['treatment'].apply(
        lambda x: 1 if str(x).lower().strip() in ['yes', 'y', '1', 'true'] else 0)

    sex_map = {'male': 1, 'm': 1, '\u7537': 1, '\u7537\u6027': 1, '1': 1,
               'female': 0, 'f': 0, '\u5973': 0, '\u5973\u6027': 0, '2': 0, '0': 0}
    df['sex_value'] = df['sex'].apply(
        lambda x: sex_map.get(str(x).lower().strip(), np.nan) if pd.notna(x) else np.nan
    )
    df['ga_value'] = pd.to_numeric(df['ga'], errors='coerce')
    df['bw_value'] = pd.to_numeric(df['bw'], errors='coerce')
    df['pma_value'] = pd.to_numeric(df['pma'], errors='coerce')

    return df


quality_df = load_quality_labeled_images(config.KUBOTA_DIR)
patient_df = load_patient_data(config.EXCEL_PATH)

dataset_df = quality_df.merge(
    patient_df[['video_id', 'zone_label', 'stage_label', 'plus_label',
                'aggressive_rop_label', 'treatment_label',
                'sex_value', 'ga_value', 'bw_value', 'pma_value']],
    on='video_id', how='left'
).dropna(subset=['zone_label'])

dataset_df = dataset_df[dataset_df['quality'].isin(config.QUALITY_FILTER)].copy()

print(f'Dataset size: {len(dataset_df)}')
print(f'Quality distribution: {dataset_df["quality"].value_counts().to_dict()}')
print(f'Unique video_ids: {dataset_df["video_id"].nunique()}')

clinical_cols = ['sex_value', 'ga_value', 'bw_value', 'pma_value']
print(f'\nClinical data completeness:')
for col in clinical_cols:
    n_valid = dataset_df[col].notna().sum()
    pct = n_valid / len(dataset_df) * 100
    print(f'  {col:12s}: {n_valid:>5}/{len(dataset_df)} ({pct:.1f}%)')

print(f'\nPlus label distribution:')
print(dataset_df['plus_label'].value_counts().sort_index())

Dataset size: 6491
Quality distribution: {'Good': 4494, 'Fair': 1997}
Unique video_ids: 348

Clinical data completeness:
  sex_value   :  6491/6491 (100.0%)
  ga_value    :  6491/6491 (100.0%)
  bw_value    :  6491/6491 (100.0%)
  pma_value   :  6491/6491 (100.0%)

Plus label distribution:
plus_label
-1      22
 0    5422
 1     639
 2     408
Name: count, dtype: int64


In [3]:
# ==================== Cell 3: Model / Loss / Dataset ====================

# --- Augmentation ---

def get_train_transforms(img_size: int = 512) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Rotate(limit=180, p=0.9, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.OneOf([
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=1.0),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
        ], p=0.7),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=5, p=1.0),
        ], p=0.3),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.85, 1.0), ratio=(0.9, 1.1), p=0.4),
        A.CoarseDropout(max_holes=8, max_height=img_size//16, max_width=img_size//16, p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def get_valid_transforms(img_size: int = 512) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


train_transforms = get_train_transforms(config.IMG_SIZE)
valid_transforms = get_valid_transforms(config.IMG_SIZE)


# --- Dataset ---

class ROPClinicalDataset(Dataset):
    """ROP Dataset with clinical feature support."""

    def __init__(self, df: pd.DataFrame, transforms: A.Compose = None,
                 clinical_mean: np.ndarray = None, clinical_std: np.ndarray = None,
                 clinical_median: np.ndarray = None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.task_columns = {
            'zone': 'zone_label',
            'stage': 'stage_label',
            'plus': 'plus_label',
            'aggressive_rop': 'aggressive_rop_label',
            'treatment': 'treatment_label'
        }
        self.clinical_continuous_cols = ['ga_value', 'bw_value', 'pma_value']
        self.clinical_mean = clinical_mean
        self.clinical_std = clinical_std
        self.clinical_median = clinical_median

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]

        image = cv2.imread(row['image_path'])
        if image is None:
            raise ValueError(f'Failed to load: {row["image_path"]}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            image = self.transforms(image=image)['image']

        labels = {}
        for task_name, col_name in self.task_columns.items():
            labels[task_name] = torch.tensor(int(row.get(col_name, -1)), dtype=torch.long)

        continuous = np.array([row[c] for c in self.clinical_continuous_cols], dtype=np.float64)

        nan_mask = np.isnan(continuous)
        if nan_mask.any() and self.clinical_median is not None:
            continuous[nan_mask] = self.clinical_median[nan_mask]

        if self.clinical_mean is not None and self.clinical_std is not None:
            std_safe = np.where(self.clinical_std == 0, 1.0, self.clinical_std)
            continuous = (continuous - self.clinical_mean) / std_safe

        sex = row.get('sex_value', np.nan)
        sex = 0.5 if pd.isna(sex) else float(sex)

        clinical = np.concatenate([continuous, [sex]]).astype(np.float32)
        clinical = torch.tensor(clinical, dtype=torch.float32)

        return {'image': image, 'labels': labels, 'clinical': clinical, 'image_path': row['image_path']}


# --- Model ---

class ClinicalEncoder(nn.Module):
    def __init__(self, n_features: int = 4, embed_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, embed_dim),
            nn.ReLU(),
            nn.BatchNorm1d(embed_dim),
        )

    def forward(self, x):
        return self.encoder(x)


class MultiTaskHead(nn.Module):
    def __init__(self, in_features: int, num_classes: int, dropout: float = 0.3):
        super().__init__()
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.head(x)


class ROPClinicalMultiTaskModel(nn.Module):
    """Multi-task model with full clinical feature fusion."""

    TASK_CONFIG = {
        'zone': (3, True),
        'stage': (4, True),
        'plus': (3, True),
        'aggressive_rop': (2, True),
        'treatment': (2, True)
    }

    CLINICAL_TASKS = {'zone', 'stage', 'plus', 'aggressive_rop', 'treatment'}

    def __init__(self, model_name: str = 'efficientnet_b0', pretrained: bool = True,
                 dropout: float = 0.3, n_clinical: int = 4, clinical_embed_dim: int = 32):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        img_features = self.backbone.num_features

        self.clinical_encoder = ClinicalEncoder(n_clinical, clinical_embed_dim)
        fused_features = img_features + clinical_embed_dim

        self.heads = nn.ModuleDict()
        for task_name, (num_classes, _) in self.TASK_CONFIG.items():
            self.heads[task_name] = MultiTaskHead(fused_features, num_classes, dropout)

        print(f'Model: {model_name}, Image features: {img_features}, '
              f'Clinical embed: {clinical_embed_dim}, Fused: {fused_features}')

    def forward(self, x, clinical):
        img_feat = self.backbone(x)
        clin_feat = self.clinical_encoder(clinical)
        fused = torch.cat([img_feat, clin_feat], dim=1)

        outputs = {}
        for task, head in self.heads.items():
            outputs[task] = head(fused)
        return outputs


# --- Loss Functions ---

class WeightedFocalLoss(nn.Module):
    def __init__(self, alpha: torch.Tensor = None, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        if self.alpha is not None:
            alpha = self.alpha.to(inputs.device)
            alpha_t = alpha[targets]
            focal_loss = alpha_t * focal_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class ClassBalancedMultiTaskLoss(nn.Module):
    def __init__(self, task_weights, class_weights, label_smoothing=0.1,
                 use_focal=True, focal_gamma=2.0):
        super().__init__()
        self.task_weights = task_weights
        self.class_weights = class_weights
        self.label_smoothing = label_smoothing
        self.use_focal = use_focal
        self.focal_gamma = focal_gamma

        self.focal_losses = nn.ModuleDict()
        for task, weights in class_weights.items():
            self.focal_losses[task] = WeightedFocalLoss(alpha=weights, gamma=focal_gamma)

    def forward(self, outputs, labels):
        total_loss = 0.0
        task_losses = {}

        for task_name, output in outputs.items():
            target = labels[task_name]
            mask = target >= 0

            if mask.sum() == 0:
                task_losses[task_name] = torch.tensor(0.0, device=output.device)
                continue

            output = output[mask]
            target = target[mask]

            weights = self.class_weights.get(task_name)
            if weights is not None:
                weights = weights.to(output.device)

            ce_loss = F.cross_entropy(output, target, weight=weights,
                                      label_smoothing=self.label_smoothing)

            if self.use_focal and task_name in self.focal_losses:
                focal_loss = self.focal_losses[task_name](output, target)
                loss = (ce_loss + focal_loss) / 2
            else:
                loss = ce_loss

            task_weight = self.task_weights.get(task_name, 1.0)
            task_losses[task_name] = loss
            total_loss += task_weight * loss

        return total_loss, task_losses


# --- MixUp ---

def mixup_data(x, clinical, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    mixed_clinical = lam * clinical + (1 - lam) * clinical[index, :]
    y_a, y_b = y, {k: v[index] for k, v in y.items()}

    return mixed_x, mixed_clinical, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    loss_a, _ = criterion(pred, y_a)
    loss_b, _ = criterion(pred, y_b)
    return lam * loss_a + (1 - lam) * loss_b


print('Model, Loss, Dataset, MixUp defined.')

Model, Loss, Dataset, MixUp defined.


In [4]:
# ==================== Cell 4: Training & Evaluation Functions ====================

def compute_metrics(preds: Dict[str, np.ndarray], labels: Dict[str, np.ndarray]) -> Dict:
    metrics = {}

    for task_name in preds.keys():
        y_pred = preds[task_name]
        y_true = labels[task_name]

        mask = y_true >= 0
        y_pred = y_pred[mask]
        y_true = y_true[mask]

        if len(y_true) == 0:
            metrics[task_name] = {'accuracy': 0.0, 'kappa': 0.0, 'f1_macro': 0.0}
            continue

        pred_classes = y_pred.argmax(axis=1) if y_pred.ndim > 1 else y_pred

        task_metrics = {
            'accuracy': accuracy_score(y_true, pred_classes),
            'kappa': cohen_kappa_score(y_true, pred_classes, weights='quadratic'),
            'f1_macro': f1_score(y_true, pred_classes, average='macro', zero_division=0),
        }

        if task_name in ['aggressive_rop', 'treatment']:
            cm = confusion_matrix(y_true, pred_classes, labels=[0, 1])
            if cm.shape == (2, 2) and (cm[1, 0] + cm[1, 1]) > 0:
                task_metrics['sensitivity'] = cm[1, 1] / (cm[1, 0] + cm[1, 1])
            else:
                task_metrics['sensitivity'] = 0.0

            try:
                probs = torch.softmax(torch.tensor(y_pred), dim=1)[:, 1].numpy()
                task_metrics['auc'] = roc_auc_score(y_true, probs)
            except:
                task_metrics['auc'] = 0.0

        metrics[task_name] = task_metrics

    return metrics


def train_epoch(model, dataloader, criterion, optimizer, scheduler, device, mixup_alpha=0.2, mixup_p=0.5):
    model.train()
    running_loss = 0.0

    for batch in tqdm(dataloader, desc='Training', leave=False):
        images = batch['image'].to(device)
        clinical = batch['clinical'].to(device)
        labels = {k: v.to(device) for k, v in batch['labels'].items()}

        if np.random.random() < mixup_p:
            images, clinical, labels_a, labels_b, lam = mixup_data(images, clinical, labels, mixup_alpha)
            outputs = model(images, clinical)
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
        else:
            outputs = model(images, clinical)
            loss, _ = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

    return running_loss / len(dataloader)


def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = {task: [] for task in ROPClinicalMultiTaskModel.TASK_CONFIG.keys()}
    all_labels = {task: [] for task in ROPClinicalMultiTaskModel.TASK_CONFIG.keys()}

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Validation', leave=False):
            images = batch['image'].to(device)
            clinical = batch['clinical'].to(device)
            labels = {k: v.to(device) for k, v in batch['labels'].items()}

            outputs = model(images, clinical)
            loss, _ = criterion(outputs, labels)
            running_loss += loss.item()

            for task in all_preds.keys():
                all_preds[task].append(outputs[task].cpu().numpy())
                all_labels[task].append(labels[task].cpu().numpy())

    for task in all_preds.keys():
        all_preds[task] = np.concatenate(all_preds[task])
        all_labels[task] = np.concatenate(all_labels[task])

    metrics = compute_metrics(all_preds, all_labels)

    return running_loss / len(dataloader), metrics, all_preds, all_labels


print('Training and evaluation functions defined.')

Training and evaluation functions defined.


In [5]:
# ==================== Cell 5: 5-Fold Train-Val-Test CV Loop ====================

dataset_df['stratify_target'] = dataset_df['zone_label'].astype(str) + '_' + dataset_df['stage_label'].astype(str)
sgkf = StratifiedGroupKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=42)

# Pre-generate all 5 folds
folds = list(sgkf.split(dataset_df, dataset_df['stratify_target'], dataset_df['video_id']))
print(f'Generated {len(folds)} folds')
for i, (train_idx, val_idx) in enumerate(folds):
    print(f'  Fold {i}: train={len(train_idx)}, val={len(val_idx)}, '
          f'train_vids={dataset_df.iloc[train_idx]["video_id"].nunique()}, '
          f'val_vids={dataset_df.iloc[val_idx]["video_id"].nunique()}')

clinical_continuous_cols = ['ga_value', 'bw_value', 'pma_value']
all_fold_results = []
all_predictions = []

for test_fold_idx in range(config.N_FOLDS):
    val_fold_idx = (test_fold_idx + 1) % config.N_FOLDS
    train_fold_indices = [i for i in range(config.N_FOLDS) if i != test_fold_idx and i != val_fold_idx]

    # Extract indices
    test_indices = folds[test_fold_idx][1]  # val_idx from StratifiedGroupKFold
    val_indices = folds[val_fold_idx][1]
    train_indices = np.concatenate([folds[i][1] for i in train_fold_indices])

    train_df = dataset_df.iloc[train_indices].reset_index(drop=True)
    val_df = dataset_df.iloc[val_indices].reset_index(drop=True)
    test_df = dataset_df.iloc[test_indices].reset_index(drop=True)

    # Verify no video_id overlap
    train_vids = set(train_df['video_id'])
    val_vids = set(val_df['video_id'])
    test_vids = set(test_df['video_id'])
    assert len(train_vids & val_vids) == 0, f'Train-Val overlap in iter {test_fold_idx+1}'
    assert len(train_vids & test_vids) == 0, f'Train-Test overlap in iter {test_fold_idx+1}'
    assert len(val_vids & test_vids) == 0, f'Val-Test overlap in iter {test_fold_idx+1}'

    fold_dir = config.OUTPUT_DIR / f'fold_{test_fold_idx + 1}'
    fold_dir.mkdir(exist_ok=True)
    best_model_path = fold_dir / 'best_model.pt'

    # Compute class weights from TRAIN only
    fold_class_weights = {}
    for task, col in [('zone', 'zone_label'), ('stage', 'stage_label'), ('plus', 'plus_label'),
                      ('aggressive_rop', 'aggressive_rop_label'), ('treatment', 'treatment_label')]:
        valid_mask = train_df[col] >= 0
        counts = train_df[valid_mask][col].value_counts().sort_index()
        n_classes = {'zone': 3, 'stage': 4, 'plus': 3, 'aggressive_rop': 2, 'treatment': 2}[task]
        full_counts = [counts.get(i, 1) for i in range(n_classes)]
        total = sum(full_counts)
        weights = [total / (n_classes * c) for c in full_counts]
        min_w = min(weights)
        weights = [w / min_w for w in weights]
        fold_class_weights[task] = torch.tensor(weights, dtype=torch.float32)

    # Clinical stats from TRAIN only
    clinical_mean = train_df[clinical_continuous_cols].mean().values.astype(np.float32)
    clinical_std = train_df[clinical_continuous_cols].std().values.astype(np.float32)
    clinical_median = train_df[clinical_continuous_cols].median().values.astype(np.float32)

    # === Skip if already trained ===
    if best_model_path.exists():
        print(f'\nIteration {test_fold_idx + 1}/{config.N_FOLDS} - SKIPPED (already trained)')
        print(f'  Test fold: {test_fold_idx}, Val fold: {val_fold_idx}, Train folds: {train_fold_indices}')
        print(f'  Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

        checkpoint = torch.load(best_model_path, map_location=device)
        clinical_mean = checkpoint['clinical_mean']
        clinical_std = checkpoint['clinical_std']
        clinical_median = checkpoint['clinical_median']

        test_dataset = ROPClinicalDataset(
            test_df, transforms=valid_transforms,
            clinical_mean=clinical_mean, clinical_std=clinical_std,
            clinical_median=clinical_median
        )
        test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

        model = ROPClinicalMultiTaskModel(
            config.MODEL_NAME, pretrained=False, dropout=config.DROPOUT,
            n_clinical=config.N_CLINICAL_FEATURES,
            clinical_embed_dim=config.CLINICAL_EMBED_DIM
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        model = model.to(device)

        criterion = ClassBalancedMultiTaskLoss(
            task_weights=config.TASK_WEIGHTS,
            class_weights=fold_class_weights,
            label_smoothing=config.LABEL_SMOOTHING,
            use_focal=config.USE_FOCAL_LOSS,
            focal_gamma=config.FOCAL_GAMMA
        )

        test_loss, final_metrics, preds, labels = validate_epoch(model, test_loader, criterion, device)

        print(f'  Test loss: {test_loss:.4f}')
        print(f'  AROP Sens: {final_metrics["aggressive_rop"].get("sensitivity", 0):.4f}')
        print(f'  Treat Sens: {final_metrics["treatment"].get("sensitivity", 0):.4f}')

        all_fold_results.append({
            'iteration': test_fold_idx + 1,
            'test_fold': test_fold_idx,
            'val_fold': val_fold_idx,
            'train_folds': train_fold_indices,
            'test_size': len(test_df),
            'test_loss': test_loss,
            'metrics': final_metrics
        })

        for i, row in test_df.iterrows():
            pred_entry = {
                'fold': test_fold_idx + 1, 'image_path': row['image_path'], 'video_id': row['video_id'],
                'ga_value': row.get('ga_value', np.nan),
                'bw_value': row.get('bw_value', np.nan),
                'pma_value': row.get('pma_value', np.nan),
                'sex_value': row.get('sex_value', np.nan),
            }
            idx = test_df.index.get_loc(i)
            for task in preds.keys():
                pred_entry[f'{task}_label'] = labels[task][idx]
                pred_entry[f'{task}_pred'] = preds[task][idx].argmax()
                if preds[task][idx].ndim > 0:
                    probs = torch.softmax(torch.tensor(preds[task][idx]), dim=0)
                    for ci in range(probs.shape[0]):
                        pred_entry[f'{task}_prob_{ci}'] = probs[ci].item()
            all_predictions.append(pred_entry)

        del model
        torch.cuda.empty_cache()
        continue

    # === Train ===
    print(f'\n{"="*60}')
    print(f'Iteration {test_fold_idx + 1}/{config.N_FOLDS}')
    print(f'  Test fold: {test_fold_idx}, Val fold: {val_fold_idx}, Train folds: {train_fold_indices}')
    print(f'  Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
    print(f'  Train videos: {len(train_vids)}, Val videos: {len(val_vids)}, Test videos: {len(test_vids)}')
    print(f'{"="*60}')

    print(f'Clinical stats (train folds {train_fold_indices}):')
    print(f'  Mean:   GA={clinical_mean[0]:.1f}, BW={clinical_mean[1]:.1f}, PMA={clinical_mean[2]:.1f}')
    print(f'  Std:    GA={clinical_std[0]:.2f}, BW={clinical_std[1]:.2f}, PMA={clinical_std[2]:.2f}')
    print(f'  Median: GA={clinical_median[0]:.1f}, BW={clinical_median[1]:.1f}, PMA={clinical_median[2]:.1f}')

    train_dataset = ROPClinicalDataset(
        train_df, transforms=train_transforms,
        clinical_mean=clinical_mean, clinical_std=clinical_std,
        clinical_median=clinical_median
    )
    val_dataset = ROPClinicalDataset(
        val_df, transforms=valid_transforms,
        clinical_mean=clinical_mean, clinical_std=clinical_std,
        clinical_median=clinical_median
    )
    test_dataset = ROPClinicalDataset(
        test_df, transforms=valid_transforms,
        clinical_mean=clinical_mean, clinical_std=clinical_std,
        clinical_median=clinical_median
    )

    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

    model = ROPClinicalMultiTaskModel(
        config.MODEL_NAME, pretrained=config.PRETRAINED, dropout=config.DROPOUT,
        n_clinical=config.N_CLINICAL_FEATURES,
        clinical_embed_dim=config.CLINICAL_EMBED_DIM
    )
    model = model.to(device)

    criterion = ClassBalancedMultiTaskLoss(
        task_weights=config.TASK_WEIGHTS,
        class_weights=fold_class_weights,
        label_smoothing=config.LABEL_SMOOTHING,
        use_focal=config.USE_FOCAL_LOSS,
        focal_gamma=config.FOCAL_GAMMA
    )

    optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    scheduler = OneCycleLR(optimizer, max_lr=config.LEARNING_RATE, epochs=config.EPOCHS, steps_per_epoch=len(train_loader))

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(config.EPOCHS):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, scheduler, device, config.MIXUP_ALPHA, config.MIXUP_P)
        val_loss, val_metrics, _, _ = validate_epoch(model, val_loader, criterion, device)

        arop_sens = val_metrics['aggressive_rop'].get('sensitivity', 0)
        treat_sens = val_metrics['treatment'].get('sensitivity', 0)

        print(f'Epoch {epoch + 1}: Val_Loss={val_loss:.4f}, AROP_Sens={arop_sens:.3f}, Treat_Sens={treat_sens:.3f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'clinical_mean': clinical_mean,
                'clinical_std': clinical_std,
                'clinical_median': clinical_median,
            }, best_model_path)
        else:
            patience_counter += 1
            if patience_counter >= config.PATIENCE:
                print(f'Early stopping at epoch {epoch + 1}')
                break

    # Reload best model and evaluate on TEST fold
    checkpoint = torch.load(best_model_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    test_loss, final_metrics, preds, labels = validate_epoch(model, test_loader, criterion, device)

    print(f'\nTest fold results (best val_loss={best_val_loss:.4f}):')
    for task, m in final_metrics.items():
        print(f'  {task}: {m}')

    all_fold_results.append({
        'iteration': test_fold_idx + 1,
        'test_fold': test_fold_idx,
        'val_fold': val_fold_idx,
        'train_folds': train_fold_indices,
        'train_size': len(train_df),
        'val_size': len(val_df),
        'test_size': len(test_df),
        'best_val_loss': best_val_loss,
        'test_loss': test_loss,
        'metrics': final_metrics
    })

    for i, row in test_df.iterrows():
        pred_entry = {
            'fold': test_fold_idx + 1, 'image_path': row['image_path'], 'video_id': row['video_id'],
            'ga_value': row.get('ga_value', np.nan),
            'bw_value': row.get('bw_value', np.nan),
            'pma_value': row.get('pma_value', np.nan),
            'sex_value': row.get('sex_value', np.nan),
        }
        idx = test_df.index.get_loc(i)
        for task in preds.keys():
            pred_entry[f'{task}_label'] = labels[task][idx]
            pred_entry[f'{task}_pred'] = preds[task][idx].argmax()
            if preds[task][idx].ndim > 0:
                probs = torch.softmax(torch.tensor(preds[task][idx]), dim=0)
                for ci in range(probs.shape[0]):
                    pred_entry[f'{task}_prob_{ci}'] = probs[ci].item()
        all_predictions.append(pred_entry)

    del model, optimizer, scheduler
    torch.cuda.empty_cache()

# Save predictions
results_df = pd.DataFrame(all_predictions)
results_df.to_csv(config.OUTPUT_DIR / 'predictions.csv', index=False)
print(f'\nPredictions saved: {len(results_df)} rows (expected ~{len(dataset_df)})')
print(f'Unique video_ids in predictions: {results_df["video_id"].nunique()}')

Generated 5 folds
  Fold 0: train=5229, val=1262, train_vids=279, val_vids=69
  Fold 1: train=5137, val=1354, train_vids=281, val_vids=67
  Fold 2: train=5055, val=1436, train_vids=279, val_vids=69
  Fold 3: train=5392, val=1099, train_vids=279, val_vids=69
  Fold 4: train=5151, val=1340, train_vids=274, val_vids=74

Iteration 1/5 - SKIPPED (already trained)
  Test fold: 0, Val fold: 1, Train folds: [2, 3, 4]
  Train: 3875, Val: 1354, Test: 1262
Model: efficientnet_b0, Image features: 1280, Clinical embed: 32, Fused: 1312


Validation:   0%|          | 0/79 [00:00<?, ?it/s]

  Test loss: 4.2223
  AROP Sens: 1.0000
  Treat Sens: 0.7848

Iteration 2/5 - SKIPPED (already trained)
  Test fold: 1, Val fold: 2, Train folds: [0, 3, 4]
  Train: 3701, Val: 1436, Test: 1354
Model: efficientnet_b0, Image features: 1280, Clinical embed: 32, Fused: 1312


Validation:   0%|          | 0/85 [00:00<?, ?it/s]

  Test loss: 5.2995
  AROP Sens: 0.1778
  Treat Sens: 0.8639

Iteration 3/5 - SKIPPED (already trained)
  Test fold: 2, Val fold: 3, Train folds: [0, 1, 4]
  Train: 3956, Val: 1099, Test: 1436
Model: efficientnet_b0, Image features: 1280, Clinical embed: 32, Fused: 1312


...

Training:   0%|          | 0/243 [00:00<?, ?it/s]

Validation:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch 29: Val_Loss=5.0487, AROP_Sens=0.000, Treat_Sens=0.430
Early stopping at epoch 29


Validation:   0%|          | 0/84 [00:00<?, ?it/s]


Test fold results (best val_loss=4.8927):
  zone: {'accuracy': 0.7261194029850746, 'kappa': 0.6149539590470028, 'f1_macro': 0.7096218095473433}
  stage: {'accuracy': 0.6253731343283582, 'kappa': 0.6269384880694557, 'f1_macro': 0.62844066139471}
  plus: {'accuracy': 0.8029850746268656, 'kappa': 0.6929863255403617, 'f1_macro': 0.5997116960571037}
  aggressive_rop: {'accuracy': 0.7276119402985075, 'kappa': 0.22469437962869032, 'f1_macro': 0.567694177438957, 'sensitivity': 0.975609756097561, 'auc': 0.9447051068284927}
  treatment: {'accuracy': 0.905223880597015, 'kappa': 0.6340813114415709, 'f1_macro': 0.8160436963780562, 'sensitivity': 0.8333333333333334, 'auc': 0.9517765724037055}

Predictions saved: 6491 rows (expected ~6491)
Unique video_ids in predictions: 348


In [6]:
# ==================== Cell 6: Aggregate Results ====================

def aggregate_metrics(fold_results):
    agg = {}
    tasks = fold_results[0]['metrics'].keys()
    for task in tasks:
        task_agg = {}
        for metric in fold_results[0]['metrics'][task].keys():
            values = [f['metrics'][task][metric] for f in fold_results]
            task_agg[metric] = (np.mean(values), np.std(values))
        agg[task] = task_agg
    return agg

agg = aggregate_metrics(all_fold_results)

print('\n' + '=' * 70)
print('TRAIN-VAL-TEST 5-FOLD CV RESULTS (Test fold predictions)')
print('=' * 70)

for task in ['zone', 'stage', 'plus', 'aggressive_rop', 'treatment']:
    print(f'\n{task.upper()}:')
    for metric, (mean, std) in agg[task].items():
        print(f'  {metric:15s}: {mean:.4f} +/- {std:.4f}')

# Save config.json
config_dict = {
    'version': config.VERSION,
    'cv_method': 'train-val-test 5-fold (val=early stopping, test=evaluation)',
    'model_name': config.MODEL_NAME,
    'quality_filter': config.QUALITY_FILTER,
    'n_clinical_features': config.N_CLINICAL_FEATURES,
    'clinical_embed_dim': config.CLINICAL_EMBED_DIM,
    'clinical_features': ['ga_value', 'bw_value', 'pma_value', 'sex_value'],
    'clinical_preprocessing': {
        'continuous': 'z-score per train folds (ga, bw, pma)',
        'binary': 'raw 0/1 (sex)',
        'missing_continuous': 'median imputation (per train folds)',
        'missing_sex': '0.5',
    },
    'use_focal_loss': config.USE_FOCAL_LOSS,
    'focal_gamma': config.FOCAL_GAMMA,
    'label_smoothing': config.LABEL_SMOOTHING,
    'results': {
        task: {metric: f'{mean:.4f} +/- {std:.4f}' for metric, (mean, std) in metrics.items()}
        for task, metrics in agg.items()
    },
    'fold_details': [
        {k: v for k, v in fr.items() if k != 'metrics'}
        for fr in all_fold_results
    ],
}

with open(config.OUTPUT_DIR / 'config.json', 'w') as f:
    json_lib.dump(config_dict, f, indent=2)

print(f'\nConfig saved to: {config.OUTPUT_DIR / "config.json"}')
print(f'Predictions: {len(results_df)} rows')


TRAIN-VAL-TEST 5-FOLD CV RESULTS (Test fold predictions)

ZONE:
  accuracy       : 0.7230 +/- 0.0640
  kappa          : 0.6295 +/- 0.0536
  f1_macro       : 0.6848 +/- 0.0482

STAGE:
  accuracy       : 0.7210 +/- 0.0514
  kappa          : 0.7362 +/- 0.0580
  f1_macro       : 0.7102 +/- 0.0457

PLUS:
  accuracy       : 0.8649 +/- 0.0445
  kappa          : 0.6937 +/- 0.0636
  f1_macro       : 0.6475 +/- 0.0723

AGGRESSIVE_ROP:
  accuracy       : 0.8342 +/- 0.0850
  kappa          : 0.2574 +/- 0.2093
  f1_macro       : 0.6027 +/- 0.1156
  sensitivity    : 0.8307 +/- 0.3266
  auc            : 0.8708 +/- 0.2289

TREATMENT:
  accuracy       : 0.9264 +/- 0.0310
  kappa          : 0.6604 +/- 0.0814
  f1_macro       : 0.8289 +/- 0.0418
  sensitivity    : 0.8302 +/- 0.0841
  auc            : 0.9431 +/- 0.0534

Config saved to: C:\Users\ykita\ROP_AI_project\Article\outputs_clinical_v3_tvt\config.json
Predictions: 6491 rows


In [7]:
# ==================== Cell 7: Quality Features Merge & Top-10 Selection ====================

# Module imports for quality assessment
sys.path.insert(0, str(Path(r'C:\Users\ykita\ROP_AI_project\ROP_project\multicenter_study')))
from quality_assessment import compute_mbss_components, compute_disc_edge_coverage, compute_mbss_score
from select_best_images import minmax_norm

EDGE_COVERAGE_CUTOFF = 0.80
WEIGHT_RETINA = 0.4
WEIGHT_GRAD = 0.4
WEIGHT_MBSS = 0.2

# Load predictions
pred_df = pd.read_csv(config.OUTPUT_DIR / 'predictions.csv')
pred_df['image_name'] = pred_df['image_path'].apply(lambda x: Path(x).name)
print(f'predictions.csv: {len(pred_df)} images, {pred_df["video_id"].nunique()} video_ids')

# Source 1: Kubota Excel
KUBOTA_EXCEL = Path(r'E:\Multicenter_ROP_study\Multicenter_images\Kubota_selection\selected_images_disc_retina.xlsx')
feat_cols = ['retina_ratio', 'mbss_Grad_p90', 'mbss_score', 'disc_detected', 'disc_edge_coverage_ratio']

if KUBOTA_EXCEL.exists():
    kubota_df = pd.read_excel(KUBOTA_EXCEL)
    avail = [c for c in feat_cols if c in kubota_df.columns]
    kubota_features = kubota_df[['image_name'] + avail].drop_duplicates(subset='image_name', keep='first')
    merged_df = pred_df.merge(kubota_features, on='image_name', how='left')
    print(f'Kubota Excel merged: {merged_df["retina_ratio"].notna().sum()}/{len(merged_df)}')
else:
    # Fallback: try existing top10 CSV from v3
    fallback_csv = Path(r'C:\Users\ykita\ROP_AI_project\ROP_project\multicenter_study\outputs_clinical_v3\top10_selected_images.csv')
    if fallback_csv.exists():
        fb_df = pd.read_csv(fallback_csv)
        fb_cols = [c for c in feat_cols if c in fb_df.columns]
        fb_features = fb_df[['image_name'] + fb_cols].drop_duplicates(subset='image_name', keep='first')
        merged_df = pred_df.merge(fb_features, on='image_name', how='left')
        print(f'Fallback CSV merged: {merged_df["retina_ratio"].notna().sum()}/{len(merged_df)}')
    else:
        merged_df = pred_df.copy()
        print('WARNING: No quality features source available!')

# Fill from top-level Excel
TOP_EXCEL = Path(r'E:\Multicenter_ROP_study\Multicenter_images\selected_images_disc_retina.xlsx')
if TOP_EXCEL.exists() and 'retina_ratio' in merged_df.columns:
    tl = pd.read_excel(TOP_EXCEL)
    tl_feat = [c for c in feat_cols if c in tl.columns and c in merged_df.columns]
    if 'image_name' in tl.columns and tl_feat:
        tl_features = tl[['image_name'] + tl_feat].drop_duplicates(subset='image_name', keep='first').set_index('image_name')
        mm = merged_df['retina_ratio'].isna()
        for col in tl_feat:
            fill = merged_df.loc[mm, 'image_name'].map(tl_features[col])
            merged_df.loc[mm, col] = fill.values
        print(f'Top-level Excel filled: {merged_df["retina_ratio"].notna().sum()}/{len(merged_df)}')

n_feat = merged_df['retina_ratio'].notna().sum() if 'retina_ratio' in merged_df.columns else 0
print(f'\nFinal features available: {n_feat}/{len(merged_df)}')


# --- Top-K Selection ---

def select_top_k_per_video(df, top_k=10, edge_cov_cutoff=0.80):
    """C-rule3_thr0.80 per video_id"""
    all_selected = []
    summary = {'total_videos': 0, 'stage1_only': 0, 'needed_fallback': 0,
               'insufficient': 0, 'no_features': 0}

    for vid, group in df.groupby('video_id'):
        summary['total_videos'] += 1
        valid = group[group['retina_ratio'].notna() & (group['retina_ratio'] > 0)].copy()
        if len(valid) == 0:
            summary['no_features'] += 1
            continue

        stage1 = valid[
            (valid['disc_detected'] == True) &
            valid['disc_edge_coverage_ratio'].notna() &
            (valid['disc_edge_coverage_ratio'] >= edge_cov_cutoff)
        ].copy()

        if len(stage1) > 0:
            stage1['retina_norm'] = minmax_norm(stage1['retina_ratio'].fillna(0))
            stage1['grad_norm'] = minmax_norm(stage1['mbss_Grad_p90'].fillna(0))
            stage1['mbss_norm'] = minmax_norm(stage1['mbss_score'].fillna(0))
            stage1['quality_score'] = (
                WEIGHT_RETINA * stage1['retina_norm'] +
                WEIGHT_GRAD * stage1['grad_norm'] +
                WEIGHT_MBSS * stage1['mbss_norm']
            )
            stage1 = stage1.sort_values('quality_score', ascending=False)
            selected = stage1.head(top_k).copy()
            selected['selection_stage'] = 'Stage1_edge_cov>=0.80'
        else:
            selected = pd.DataFrame()

        n_remaining = top_k - len(selected)
        if n_remaining > 0:
            used_idx = selected.index if len(selected) > 0 else pd.Index([])
            remaining = valid[~valid.index.isin(used_idx)].copy()
            if len(remaining) > 0:
                remaining = remaining.sort_values('retina_ratio', ascending=False)
                fallback = remaining.head(n_remaining).copy()
                fallback['selection_stage'] = 'Stage2_fallback'
                selected = pd.concat([selected, fallback])

            if len(selected) < top_k:
                summary['insufficient'] += 1
            elif n_remaining > 0:
                summary['needed_fallback'] += 1
            else:
                summary['stage1_only'] += 1
        else:
            summary['stage1_only'] += 1

        if len(selected) > 0:
            selected['rank'] = range(1, len(selected) + 1)
            all_selected.append(selected)

    top_df = pd.concat(all_selected, ignore_index=True) if all_selected else pd.DataFrame()
    return top_df, summary


top10_df, sel_summary_10 = select_top_k_per_video(merged_df, top_k=10, edge_cov_cutoff=EDGE_COVERAGE_CUTOFF)
top5_df, sel_summary_5 = select_top_k_per_video(merged_df, top_k=5, edge_cov_cutoff=EDGE_COVERAGE_CUTOFF)

print(f'\nTop-10: {len(top10_df)} images, {top10_df["video_id"].nunique()} videos')
print(f'  Stage1 only: {sel_summary_10["stage1_only"]}, Fallback: {sel_summary_10["needed_fallback"]}')
print(f'Top-5:  {len(top5_df)} images, {top5_df["video_id"].nunique()} videos')
print(f'  Stage1 only: {sel_summary_5["stage1_only"]}, Fallback: {sel_summary_5["needed_fallback"]}')

# Save top10 CSV
save_cols = [c for c in ['video_id', 'image_name', 'image_path', 'fold', 'rank', 'selection_stage',
    'retina_ratio', 'mbss_Grad_p90', 'mbss_score', 'disc_edge_coverage_ratio',
    'zone_label', 'zone_pred', 'stage_label', 'stage_pred',
    'plus_label', 'plus_pred', 'treatment_label', 'treatment_pred',
    'aggressive_rop_label', 'aggressive_rop_pred'] if c in top10_df.columns]
top10_df[save_cols].to_csv(config.OUTPUT_DIR / 'top10_selected_images.csv', index=False)
print(f'Top-10 CSV saved: {config.OUTPUT_DIR / "top10_selected_images.csv"}')

Error reading from C:\Users\ykita\AppData\Roaming\Ultralytics\settings.json: "No Ultralytics setting 'openvino_msg'. \nView Ultralytics Settings with 'yolo settings' or at 'C:\\Users\\ykita\\AppData\\Roaming\\Ultralytics\\settings.json'\nUpdate Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings."
Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\ykita\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
predictions.csv: 6491 images, 348 video_ids
Kubota Excel merged: 6225/6491
Top-level Excel filled: 6491/6491

Final features available: 6491/6491

Top-10: 3103 images, 348 videos
  Stage1 only: 264, Fallback: 11
Top-5:  1654 images, 348 videos
  Stage1 only: 302, Fal

In [8]:
# ==================== Cell 8: Video-Level Aggregation ====================

def aggregate_hard_vote(group, pred_col):
    mode_result = stats.mode(group[pred_col], keepdims=True)
    return int(mode_result.mode[0])


def aggregate_per_video(df, method='soft'):
    """video_id-level aggregation via hard/soft vote."""
    rows = []
    for vid, group in df.groupby('video_id'):
        row = {'video_id': vid, 'n_images': len(group)}

        # Ground truth (same for all images in video)
        for col in ['zone_label', 'stage_label', 'plus_label', 'aggressive_rop_label', 'treatment_label']:
            row[col] = int(group[col].iloc[0])

        if method == 'hard':
            for col in ['zone_pred', 'stage_pred', 'plus_pred', 'aggressive_rop_pred', 'treatment_pred']:
                row[col] = aggregate_hard_vote(group, col)
            for col in ['aggressive_rop_prob_1', 'treatment_prob_1', 'zone_prob_0', 'stage_prob_3', 'plus_prob_2']:
                row[col] = float(group[col].mean())
        else:
            zone_probs = group[['zone_prob_0', 'zone_prob_1', 'zone_prob_2']].mean()
            row['zone_pred'] = int(np.argmax(zone_probs.values))
            row['zone_prob_0'] = float(zone_probs['zone_prob_0'])

            stage_probs = group[['stage_prob_0', 'stage_prob_1', 'stage_prob_2', 'stage_prob_3']].mean()
            row['stage_pred'] = int(np.argmax(stage_probs.values))
            row['stage_prob_3'] = float(stage_probs['stage_prob_3'])

            plus_probs = group[['plus_prob_0', 'plus_prob_1', 'plus_prob_2']].mean()
            row['plus_pred'] = int(np.argmax(plus_probs.values))
            row['plus_prob_2'] = float(plus_probs['plus_prob_2'])

            arop_prob = float(group['aggressive_rop_prob_1'].mean())
            row['aggressive_rop_prob_1'] = arop_prob
            row['aggressive_rop_pred'] = int(arop_prob >= 0.5)

            treat_prob = float(group['treatment_prob_1'].mean())
            row['treatment_prob_1'] = treat_prob
            row['treatment_pred'] = int(treat_prob >= 0.5)

        rows.append(row)
    return pd.DataFrame(rows)


def compute_rw_rop(df):
    rw_true = ((df['plus_label'] == 2) | (df['stage_label'] == 3) | (df['zone_label'] == 0)).astype(int)
    rw_pred = ((df['plus_pred'] == 2) | (df['stage_pred'] == 3) | (df['zone_pred'] == 0)).astype(int)
    rw_prob = 1 - ((1 - df['plus_prob_2']) * (1 - df['stage_prob_3']) * (1 - df['zone_prob_0']))
    return rw_true, rw_pred, rw_prob


def compute_binary_metrics_mv(y_true, y_pred, y_prob=None):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    result = {
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'PPV': tp / (tp + fp) if (tp + fp) > 0 else 0,
        'NPV': tn / (tn + fn) if (tn + fn) > 0 else 0,
        'F1': 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0,
    }
    if y_prob is not None and len(set(y_true)) >= 2:
        result['AUC'] = roc_auc_score(y_true, y_prob)
    return result


def evaluate_all_tasks_mv(df):
    results = {}
    results['zone'] = {
        'accuracy': accuracy_score(df['zone_label'], df['zone_pred']),
        'kappa': cohen_kappa_score(df['zone_label'], df['zone_pred']),
        'f1_macro': f1_score(df['zone_label'], df['zone_pred'], average='macro', zero_division=0),
    }
    results['stage'] = {
        'accuracy': accuracy_score(df['stage_label'], df['stage_pred']),
        'kappa': cohen_kappa_score(df['stage_label'], df['stage_pred']),
        'f1_macro': f1_score(df['stage_label'], df['stage_pred'], average='macro', zero_division=0),
    }
    results['plus'] = {
        'accuracy': accuracy_score(df['plus_label'], df['plus_pred']),
        'kappa': cohen_kappa_score(df['plus_label'], df['plus_pred']),
        'f1_macro': f1_score(df['plus_label'], df['plus_pred'], average='macro', zero_division=0),
    }
    results['aggressive_rop'] = compute_binary_metrics_mv(
        df['aggressive_rop_label'], df['aggressive_rop_pred'], df['aggressive_rop_prob_1'])
    results['treatment'] = compute_binary_metrics_mv(
        df['treatment_label'], df['treatment_pred'], df['treatment_prob_1'])
    rw_true, rw_pred, rw_prob = compute_rw_rop(df)
    results['rw_rop'] = compute_binary_metrics_mv(rw_true, rw_pred, rw_prob)
    return results


# Run all aggregations
conditions = {}
for method in ['hard', 'soft']:
    for label, src_df in [('All', merged_df), ('Top-10', top10_df), ('Top-5', top5_df)]:
        key = f'{label}_{method}'
        agg_df = aggregate_per_video(src_df, method=method)
        conditions[key] = {
            'agg_df': agg_df,
            'results': evaluate_all_tasks_mv(agg_df),
            'n_videos': len(agg_df),
        }
        print(f'{key}: {len(agg_df)} videos')

# Display comparison
print('\n' + '=' * 100)
print('Majority Vote Results (Video-level)')
print('=' * 100)
print(f'{"Task":<15} {"Metric":<12} ' +
      ' '.join(f'{k:>12}' for k in conditions.keys()))
print('-' * 100)

for task in ['zone', 'stage', 'plus']:
    for metric in ['accuracy', 'kappa', 'f1_macro']:
        vals = [conditions[k]['results'][task][metric] for k in conditions.keys()]
        print(f'{task:<15} {metric:<12} ' + ' '.join(f'{v:>12.4f}' for v in vals))
    print()

for task in ['aggressive_rop', 'treatment', 'rw_rop']:
    for metric in ['sensitivity', 'specificity', 'F1', 'AUC']:
        vals = []
        for k in conditions.keys():
            v = conditions[k]['results'][task].get(metric, float('nan'))
            vals.append(v)
        print(f'{task:<15} {metric:<12} ' + ' '.join(f'{v:>12.4f}' for v in vals))
    print()

# Save majority vote results
mv_save = {}
for key, cond in conditions.items():
    mv_save[key] = {
        'n_videos': cond['n_videos'],
        'results': {task: {k: float(v) if isinstance(v, (np.floating, float)) else int(v)
                           for k, v in metrics.items()}
                    for task, metrics in cond['results'].items()}
    }

with open(config.OUTPUT_DIR / 'majority_vote_results.json', 'w') as f:
    json_lib.dump(mv_save, f, indent=2)
print(f'\nSaved: {config.OUTPUT_DIR / "majority_vote_results.json"}')

All_hard: 348 videos
Top-10_hard: 348 videos
Top-5_hard: 348 videos
All_soft: 348 videos
Top-10_soft: 348 videos
Top-5_soft: 348 videos

Majority Vote Results (Video-level)
Task            Metric           All_hard  Top-10_hard   Top-5_hard     All_soft  Top-10_soft   Top-5_soft
----------------------------------------------------------------------------------------------------
zone            accuracy           0.7529       0.7557       0.7443       0.7443       0.7500       0.7529
zone            kappa              0.5690       0.5750       0.5588       0.5564       0.5667       0.5724
zone            f1_macro           0.7242       0.7264       0.7183       0.7173       0.7218       0.7273

stage           accuracy           0.7241       0.7126       0.7155       0.7241       0.7184       0.7213
stage           kappa              0.6145       0.5996       0.6028       0.6151       0.6078       0.6122
stage           f1_macro           0.5730       0.5607       0.5578       0.5730   

In [9]:
# ==================== Cell 9: Threshold Optimization ====================

def find_optimal_thresholds(y_true, y_prob, target_sensitivity=0.95):
    if len(set(y_true)) < 2:
        return None
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    auc_val = roc_auc_score(y_true, y_prob)

    # Sensitivity >= target
    valid_idx = np.where(tpr >= target_sensitivity)[0]
    if len(valid_idx) > 0:
        best_idx = valid_idx[np.argmin(fpr[valid_idx])]
        thresh_sens = float(thresholds[best_idx])
    else:
        thresh_sens = float(thresholds[0])

    # Youden's J
    youden_j = tpr - fpr
    youden_idx = np.argmax(youden_j)
    thresh_youden = float(thresholds[youden_idx])

    return {
        'auc': auc_val,
        'sens_95': {'threshold': thresh_sens},
        'youden': {'threshold': thresh_youden},
    }


def evaluate_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        'threshold': threshold,
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'ppv': tp / (tp + fp) if (tp + fp) > 0 else 0,
        'npv': tn / (tn + fn) if (tn + fn) > 0 else 0,
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }


# Per-image threshold optimization (using all test predictions pooled)
print('=' * 70)
print('Threshold Optimization (Per-Image, All Test Predictions)')
print('=' * 70)

# Treatment
treat_true = merged_df['treatment_label'].values.astype(int)
treat_prob = merged_df['treatment_prob_1'].values
opt_treat = find_optimal_thresholds(treat_true, treat_prob)

# RW-ROP
rw_true, _, rw_prob = compute_rw_rop(merged_df)
opt_rw = find_optimal_thresholds(rw_true, rw_prob)

for task_name, y_t, y_p, opt in [('Treatment', treat_true, treat_prob, opt_treat),
                                   ('RW-ROP', rw_true, rw_prob, opt_rw)]:
    if opt is None:
        continue
    print(f'\n--- {task_name} (AUC={opt["auc"]:.4f}) ---')
    for strat_name in ['default_0.5', 'sens_95', 'youden']:
        thr = 0.5 if strat_name == 'default_0.5' else opt[strat_name]['threshold']
        m = evaluate_at_threshold(y_t, y_p, thr)
        print(f'  {strat_name:>12s} (thr={thr:.4f}): '
              f'sens={m["sensitivity"]:.4f}, spec={m["specificity"]:.4f}, '
              f'ppv={m["ppv"]:.4f}, npv={m["npv"]:.4f}')

# Majority vote threshold optimization (Soft vote, Top-10)
print('\n' + '=' * 70)
print('Threshold Optimization (Soft Vote, Video-level)')
print('=' * 70)

for label in ['All_soft', 'Top-10_soft', 'Top-5_soft']:
    agg_df = conditions[label]['agg_df']
    rw_true_v, _, rw_prob_v = compute_rw_rop(agg_df)
    print(f'\n--- {label} ---')

    for task_name, y_t, y_p in [('Treatment', agg_df['treatment_label'], agg_df['treatment_prob_1']),
                                 ('RW-ROP', rw_true_v, rw_prob_v)]:
        opt = find_optimal_thresholds(y_t, y_p)
        if opt is None:
            continue
        print(f'  {task_name} (AUC={opt["auc"]:.4f}):')
        for strat_name in ['default_0.5', 'sens_95', 'youden']:
            thr = 0.5 if strat_name == 'default_0.5' else opt[strat_name]['threshold']
            m = evaluate_at_threshold(y_t, y_p, thr)
            print(f'    {strat_name:>12s} (thr={thr:.4f}): '
                  f'sens={m["sensitivity"]:.4f}, spec={m["specificity"]:.4f}')

# Save threshold results
threshold_results = {}
for task_name, y_t, y_p in [('treatment', treat_true, treat_prob), ('rw_rop', rw_true, rw_prob)]:
    opt = find_optimal_thresholds(y_t, y_p)
    if opt is None:
        continue
    threshold_results[task_name] = {'auc': round(opt['auc'], 4)}
    for strat in ['default_0.5', 'sens_95', 'youden']:
        thr = 0.5 if strat == 'default_0.5' else opt[strat]['threshold']
        m = evaluate_at_threshold(y_t, y_p, thr)
        threshold_results[task_name][strat] = {k: round(v, 4) for k, v in m.items()}

with open(config.OUTPUT_DIR / 'top10_evaluation_results.json', 'w') as f:
    json_lib.dump({
        'per_image_thresholds': threshold_results,
        'selection_summary_10': sel_summary_10,
        'selection_summary_5': sel_summary_5,
    }, f, indent=2)
print(f'\nSaved: {config.OUTPUT_DIR / "top10_evaluation_results.json"}')

Threshold Optimization (Per-Image, All Test Predictions)

--- Treatment (AUC=0.9502) ---
   default_0.5 (thr=0.5000): sens=0.8293, spec=0.9355, ppv=0.5791, npv=0.9809
       sens_95 (thr=0.3659): sens=0.9506, spec=0.8267, ppv=0.3697, npv=0.9936
        youden (thr=0.4072): sens=0.9171, spec=0.8915, ppv=0.4748, npv=0.9902

--- RW-ROP (AUC=0.9126) ---
   default_0.5 (thr=0.5000): sens=0.8765, spec=0.8055, ppv=0.5572, npv=0.9589
       sens_95 (thr=0.3265): sens=0.9506, spec=0.4030, ppv=0.3078, npv=0.9669
        youden (thr=0.5524): sens=0.8419, spec=0.8680, ppv=0.6404, npv=0.9516

Threshold Optimization (Soft Vote, Video-level)

--- All_soft ---
  Treatment (AUC=0.9707):
     default_0.5 (thr=0.5000): sens=0.9091, spec=0.9524
         sens_95 (thr=0.4440): sens=0.9697, spec=0.9175
          youden (thr=0.4978): sens=0.9394, spec=0.9524
  RW-ROP (AUC=0.9243):
     default_0.5 (thr=0.5000): sens=0.8929, spec=0.7614
         sens_95 (thr=0.4156): sens=0.9524, spec=0.5758
          youden (

In [10]:
# ==================== Cell 10: Vote Concordance Analysis ====================

def compute_vote_concordance(df, tasks, top_k_label):
    """
    For each video, compute the vote concordance (majority class ratio).
    Then analyze accuracy by concordance bin.
    """
    results = {}
    for task in tasks:
        pred_col = f'{task}_pred'
        label_col = f'{task}_label'
        if pred_col not in df.columns:
            continue

        vid_data = []
        for vid, group in df.groupby('video_id'):
            preds = group[pred_col].values
            true_label = int(group[label_col].iloc[0])
            n = len(preds)
            if n == 0:
                continue

            # Majority class count
            mode_result = stats.mode(preds, keepdims=True)
            majority_count = mode_result.count[0]
            majority_pred = int(mode_result.mode[0])
            concordance = majority_count / n

            vid_data.append({
                'video_id': vid,
                'n_images': n,
                'majority_pred': majority_pred,
                'true_label': true_label,
                'concordance': concordance,
                'correct': int(majority_pred == true_label),
            })

        if not vid_data:
            continue

        vid_df = pd.DataFrame(vid_data)

        # Bin by concordance
        bins = [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.01]
        bin_labels = ['<50%', '50-60%', '60-70%', '70-80%', '80-90%', '90-100%']
        vid_df['conc_bin'] = pd.cut(vid_df['concordance'], bins=bins, labels=bin_labels, right=False)

        bin_stats = []
        for b in bin_labels:
            subset = vid_df[vid_df['conc_bin'] == b]
            if len(subset) == 0:
                continue
            bin_stats.append({
                'bin': b,
                'n_videos': len(subset),
                'accuracy': subset['correct'].mean(),
                'mean_concordance': subset['concordance'].mean(),
            })

        results[task] = {
            'overall_accuracy': vid_df['correct'].mean(),
            'mean_concordance': vid_df['concordance'].mean(),
            'n_videos': len(vid_df),
            'bin_stats': bin_stats,
        }

    return results


tasks = ['zone', 'stage', 'plus', 'aggressive_rop', 'treatment']

print('=' * 80)
print('Vote Concordance Analysis')
print('=' * 80)

for top_k_label, src_df in [('Top-10', top10_df), ('Top-5', top5_df)]:
    print(f'\n--- {top_k_label} ---')
    conc_results = compute_vote_concordance(src_df, tasks, top_k_label)

    for task, res in conc_results.items():
        print(f'\n  {task} (n={res["n_videos"]} videos, '
              f'mean concordance={res["mean_concordance"]:.3f}, '
              f'overall acc={res["overall_accuracy"]:.3f}):')
        print(f'    {"Bin":<12} {"n_videos":>8} {"accuracy":>10} {"mean_conc":>10}')
        for bs in res['bin_stats']:
            print(f'    {bs["bin"]:<12} {bs["n_videos"]:>8} {bs["accuracy"]:>10.3f} {bs["mean_concordance"]:>10.3f}')

# RW-ROP concordance (derived task)
print('\n--- RW-ROP (derived) ---')
for top_k_label, src_df in [('Top-10', top10_df), ('Top-5', top5_df)]:
    # Add rw_rop columns
    src_copy = src_df.copy()
    rw_true, rw_pred, _ = compute_rw_rop(src_copy)
    src_copy['rw_rop_label'] = rw_true.values
    src_copy['rw_rop_pred'] = rw_pred.values
    conc = compute_vote_concordance(src_copy, ['rw_rop'], top_k_label)
    if 'rw_rop' in conc:
        res = conc['rw_rop']
        print(f'\n  {top_k_label} RW-ROP (n={res["n_videos"]}, '
              f'mean conc={res["mean_concordance"]:.3f}, '
              f'overall acc={res["overall_accuracy"]:.3f}):')
        print(f'    {"Bin":<12} {"n_videos":>8} {"accuracy":>10}')
        for bs in res['bin_stats']:
            print(f'    {bs["bin"]:<12} {bs["n_videos"]:>8} {bs["accuracy"]:>10.3f}')

Vote Concordance Analysis

--- Top-10 ---

  zone (n=348 videos, mean concordance=0.914, overall acc=0.756):
    Bin          n_videos   accuracy  mean_conc
    50-60%              8      0.250      0.500
    60-70%             24      0.667      0.617
    70-80%             25      0.600      0.712
    80-90%             29      0.621      0.813
    90-100%           262      0.809      0.984

  stage (n=348 videos, mean concordance=0.903, overall acc=0.713):
    Bin          n_videos   accuracy  mean_conc
    <50%                4      0.250      0.383
    50-60%              7      0.143      0.500
    60-70%             24      0.458      0.612
    70-80%             32      0.594      0.713
    80-90%             29      0.552      0.810
    90-100%           252      0.794      0.985

  plus (n=348 videos, mean concordance=0.941, overall acc=0.876):
    Bin          n_videos   accuracy  mean_conc
    <50%                1      0.000      0.400
    50-60%             12      0.667

In [11]:
# ==================== Cell 11: Results Summary ====================

print('=' * 80)
print('FINAL RESULTS SUMMARY: clinical_v3_tvt (Train-Val-Test)')
print('=' * 80)

# Per-image metrics (test fold predictions)
print('\n--- Per-Image Metrics (Test Fold Predictions) ---')
for task in ['zone', 'stage', 'plus', 'aggressive_rop', 'treatment']:
    print(f'\n  {task.upper()}:')
    for metric, (mean, std) in agg[task].items():
        print(f'    {metric:15s}: {mean:.4f} +/- {std:.4f}')

# Majority vote key results (Soft vote, Top-10)
print('\n--- Video-Level (Soft Vote, Top-10) ---')
key = 'Top-10_soft'
for task in ['zone', 'stage', 'plus', 'aggressive_rop', 'treatment', 'rw_rop']:
    print(f'\n  {task.upper()}:')
    for metric, val in conditions[key]['results'][task].items():
        if isinstance(val, (float, np.floating)):
            print(f'    {metric:15s}: {val:.4f}')

# Comparison with clinical_v3 (val=test)
print('\n\n' + '=' * 80)
print('COMPARISON: clinical_v3 (val=test) vs clinical_v3_tvt (train-val-test)')
print('=' * 80)
print('\nNote: clinical_v3 results are from the original notebook where val set')
print('was used for both early stopping and final evaluation (optimistic bias).')
print('clinical_v3_tvt uses a separate test set for final evaluation.')
print('\nLoad clinical_v3 config.json to compare:')

v3_config_path = Path(r'C:\Users\ykita\ROP_AI_project\ROP_project\multicenter_study\outputs_clinical_v3\config.json')
if v3_config_path.exists():
    with open(v3_config_path) as f:
        v3_config = json_lib.load(f)
    v3_results = v3_config.get('results', {})

    print(f'\n{"Task":<18} {"Metric":<15} {"v3 (val=test)":>15} {"v3_tvt (test)":>15} {"Delta":>10}')
    print('-' * 75)

    for task in ['zone', 'stage', 'plus', 'aggressive_rop', 'treatment']:
        if task not in v3_results or task not in agg:
            continue
        for metric in agg[task].keys():
            if metric not in v3_results[task]:
                continue
            v3_str = v3_results[task][metric]
            v3_mean = float(v3_str.split(' +/- ')[0])
            tvt_mean, tvt_std = agg[task][metric]
            delta = tvt_mean - v3_mean
            print(f'{task:<18} {metric:<15} {v3_mean:>15.4f} {tvt_mean:>12.4f}+/-{tvt_std:.4f} {delta:>+10.4f}')
        print()
else:
    print('  clinical_v3 config.json not found. Manual comparison needed.')

print('\nDone.')

FINAL RESULTS SUMMARY: clinical_v3_tvt (Train-Val-Test)

--- Per-Image Metrics (Test Fold Predictions) ---

  ZONE:
    accuracy       : 0.7230 +/- 0.0640
    kappa          : 0.6295 +/- 0.0536
    f1_macro       : 0.6848 +/- 0.0482

  STAGE:
    accuracy       : 0.7210 +/- 0.0514
    kappa          : 0.7362 +/- 0.0580
    f1_macro       : 0.7102 +/- 0.0457

  PLUS:
    accuracy       : 0.8649 +/- 0.0445
    kappa          : 0.6937 +/- 0.0636
    f1_macro       : 0.6475 +/- 0.0723

  AGGRESSIVE_ROP:
    accuracy       : 0.8342 +/- 0.0850
    kappa          : 0.2574 +/- 0.2093
    f1_macro       : 0.6027 +/- 0.1156
    sensitivity    : 0.8307 +/- 0.3266
    auc            : 0.8708 +/- 0.2289

  TREATMENT:
    accuracy       : 0.9264 +/- 0.0310
    kappa          : 0.6604 +/- 0.0814
    f1_macro       : 0.8289 +/- 0.0418
    sensitivity    : 0.8302 +/- 0.0841
    auc            : 0.9431 +/- 0.0534

--- Video-Level (Soft Vote, Top-10) ---

  ZONE:
    accuracy       : 0.7500
    kappa    